

# Some notes on using einx 

These notes are from following Ron's implementation of RoPE which use several einx tricks.

The comprehensive einx docs can be found here:

https://einx.readthedocs.io/en/stable/gettingstarted/tutorial_notation.html

TODO: Review this useful blurb from the `einx` tutorial about unnamed dimensions
```
x = np.ones((2, 1, 3))
einx.rearrange("a 1 b -> 1 1 a b 1 5 6", x).shape
(1, 1, 2, 3, 1, 5, 6)
```

### TODO:
- Insert example of a 2D square sparse matrix (d_k, d_k) operating on vectors of len d_k 
where the specifc d_k dimension is an arbtrary position in the input matrix.
(... a .... b ... ) 


In [ ]:
# Imports
import einx
import torch

## The rearrange command

In [ ]:


# create a sample vector in torch:
tt4 = torch.arange(4)
print(tt4)
einx.rearrange("... (d c) -> ... d c",tt4, d=2)

In [ ]:
# Define a nice, simple rank 1 tensor to work with
tt6 = torch.arange(6)
tt6

In [ ]:

einx.rearrange("... (d 1) -> ... d 1", tt6)

In [ ]:
# rearrange the rank 1 tensor to be of rank 2
einx.rearrange("... (d c) -> ... d c",tt6, c=2)
# a
einx.rearrange("... (number_of_pairs even_odd) -> ... number_of_pairs even_odd",tt6, even_odd=2)

In [ ]:
# rearrange the rank 1 tensor to be of rank 2
# einx.rearrange("... (d c) -> ... d c",tt6, c=3)
# a
einx.rearrange("... (number_of_pairs even_odd) -> ...  even_odd number_of_pairs",tt6, even_odd=2)

In [ ]:
# to invert the above operation:
print(tt6)
pairs = einx.rearrange("... (d c) -> ... d c",tt6, c=2)
print(pairs)
undone = einx.rearrange("... d c -> ... (d c)", pairs)
print(undone)

In [ ]:
# to invert the above operation:
print(tt6)
pairs = einx.rearrange("... (d c) -> ... d c",tt6, c=2)
print(pairs)
undone = einx.rearrange("... d c -> ... (d c)", pairs)
print(undone)

In [ ]:
tt12 = torch.arange(12)
tt12

In [ ]:
# The rearrange from rank 1 to rank 2, requires specificaion of at least one parameter, 
# how many blocks to break the input sequence into, of the size of each block ..
# ... blocks are consecutive entries of the input tensor.

einx.rearrange("... (num_blocks block_size) -> ... num_blocks block_size",tt12, block_size=6)

In [ ]:
# the above is equivalent to 
einx.rearrange("... (num_blocks block_size) -> ... num_blocks block_size",tt12, num_blocks=2)

In [ ]:
einx.rearrange("... (num_blocks block_size) -> ... num_blocks block_size",tt12, block_size=2)

### Now lets try to apply this to RoPE

In [ ]:
# set some parameters to make a concrete example ... follow the values in the test
d_k = 128
max_seq_len = 12
device = None
THETA = 10000  # typical value for small models

#### Create tables of cosines and sines, indexed by i, k.

Here $i$ is the sequence position and $k$ is the index of dimension-pairs within a token-subspace-vector

In [ ]:

k = torch.arange(d_k//2, device=device)  # subspace dimensions; shape (d_k//2,)
i = torch.arange(max_seq_len, device=device)   # positions; shape (max_seq_len,)

# reshape to make rectangles
k = einx.rearrange("a -> 1 a",k)  # shape(1,d_k//2)
i = einx.rearrange("a -> a 1",i)  # shape(max_seq_len,1)

assert isinstance(k, torch.Tensor) # just to make VS Code not complain
assert isinstance(i, torch.Tensor) # just to make VS Code not complain

# theta_{i,k} = i / (THETA^{2k/d_k})
exponents = (2./d_k) * k   # step from zero to 1 in k steps
theta_i_k = i / (THETA**exponents)  # outer product is applied here
cos_theta_i_k = torch.cos(theta_i_k)
sin_theta_i_k = torch.sin(theta_i_k)

print(f"Shape of angle, cos and sine tables are {cos_theta_i_k.shape}")
print(f"Shape of angle, cos and sine tables are {cos_theta_i_k.shape}")


# shorthand
cos_ = cos_theta_i_k
sin_ = sin_theta_i_k


In [ ]:
# lets take a little vector of length 6, in sequence position 1
D = 6
tsv = 1.0 * torch.arange(D)  # sample token-subspace-vector
print(f"tsv = {tsv}")
pairs = einx.rearrange("... (d c) -> ... d c",tsv, c=2)
print(f"pairs = \n{pairs} \n {pairs.shape}")
print(f"\n norm {torch.norm(pairs, dim=1)}")




In [ ]:
# rotate these vectors 
# 
i = 1  # set the token index in the sequence
# k from 0 to 2 (for the little 3 row example ) , runs up to d_k//2

rotated = torch.zeros(pairs.shape)
k = 0  # The first pair, and first rotation angle -- this also corresponds to the row of the "pairs" array
rotated[k, 0] = cos_theta_i_k[i, k] * pairs[k, 0] + sin_theta_i_k[i, k] * pairs[k, 1]
rotated[k, 1] = -sin_theta_i_k[i, k] * pairs[k, 0] + cos_theta_i_k[i, k] * pairs[k, 1]

k = 1  # The second pair, and second rotation angle
rotated[k, 0] = cos_theta_i_k[i, k] * pairs[k, 0] + sin_theta_i_k[i, k] * pairs[k, 1]
rotated[k, 1] = -sin_theta_i_k[i, k] * pairs[k, 0] + cos_theta_i_k[i, k] * pairs[k, 1]

k = 2  # The third pair, and third rotation angle
rotated[k, 0] = cos_theta_i_k[i, k] * pairs[k, 0] + sin_theta_i_k[i, k] * pairs[k, 1]
rotated[k, 1] = -sin_theta_i_k[i, k] * pairs[k, 0] + cos_theta_i_k[i, k] * pairs[k, 1] 

print(rotated)

In [ ]:
# sanity check that the norms are preserved
torch.norm(rotated, dim=1)

We could then do thie Rope transformation fairly efficiently by using direct multiplication (no need to matrix multiply).

Alas, for-looping is slow, so we need to learn to do this in one fell swoop. 

We will also need to make the initial vector have more than one mode ... there will be a mode for sequence position, and one for token subspace, and then more for head and batch ... 


In [ ]:
tsv = 1.0 * torch.arange(D)  # sample token-subspace-vector
tsv2 = (torch.arange(D) )   # sample token-subspace-vector
tsv3 = (torch.arange(D) + 12)   # sample token-subspace-vector
tsv4 = (torch.arange(D) + 18)   # sample token-subspace-vector

# small sequence length of 2
tsv_seq = torch.stack([tsv, tsv2, tsv3, tsv4], dim=0)  # shape (2, D)
print(f"tsv_seq = \n{tsv_seq}, {tsv_seq.shape}")

# note that the 4 rows correspond to sequence positions 0,1,2,3
# and the 6 columns correspond to the token subspace vector

In [ ]:
pairs = einx.rearrange("... (d c) -> ... d c",tsv_seq, c=2)
print(f"pairs = \n{pairs} \n {pairs.shape}")

# in this reshaping, the first mode is sequence position, the second 
# mode is pair index, the third mode is even/odd within pair

In [ ]:
rotated = torch.zeros(pairs.shape)
for i in range(pairs.shape[0]):  # over sequence length
    for k in range(pairs.shape[1]):  # over pairs
        rotated[i, k, 0] = cos_theta_i_k[i, k] * pairs[i, k, 0] + sin_theta_i_k[i, k] * pairs[i, k, 1]
        rotated[i, k, 1] = -sin_theta_i_k[i, k] * pairs[i, k, 0] + cos_theta_i_k[i, k] * pairs[i, k, 1]

print(rotated)

In [ ]:
# OK, now, how to perform this without loops:
cos_theta_i_k_4_6 = cos_theta_i_k[:4, :3]
sin_theta_i_k_4_6 = sin_theta_i_k[:4, :3]
rotated2 = torch.empty_like(pairs)
rotated2[..., 0] = cos_theta_i_k_4_6 * pairs[..., 0] + sin_theta_i_k_4_6 * pairs[..., 1]
rotated2[..., 1] = -sin_theta_i_k_4_6 * pairs[..., 0] + cos_theta_i_k_4_6 * pairs[..., 1]

print(rotated2)

In [ ]:
import pandas as pd

In [ ]:
rotated = torch.zeros(pairs.shape)
i = 0
row = 0
k = 0  # The first pair, and first rotation angle
rotated[row, 0] = cos_[i, k] * pairs[row, 0] + sin_[i, k] * pairs[row, 1]
rotated[row, 1] = -sin_[i, k] * pairs[row, 0] + cos_[i, k] * pairs[row, 1]

row = 1
k = 1  # The second pair, and first rotation angle
rotated[row, 0] = cos_theta_i_k[0, k] * pairs[row, 0] + sin_theta_i_k[i, k] * pairs[row, 1]
rotated[row, 1] = -sin_theta_i_k[0, k] * pairs[row, 0] + cos_theta_i_k[i, k] * pairs[row, 1]

row = 2
k = 2  # The second pair, and first rotation angle
rotated[row, 0] = cos_theta_i_k[0, 1] * pairs[row, 0] + sin_theta_i_k[0, 1] * pairs[row, 1]
rotated[row, 1] = -sin_theta_i_k[0, 1] * pairs[row, 0] + cos_theta_i_k[0, 1] * pairs[row, 1]

print(rotated)

In [ ]:

#einx.rearrange("... (1 1 1 a) -> ... (1 a 1)",tt12)  # shape(1,d_k//2)
#i = einx.rearrange("a -> a 1",i)  # shape(max_seq_len,1)

In [ ]:
einx.rearrange("... (d c) -> ... d c",tt6, c=2)

In [ ]:
# Specific example of wrangling a pytorch tensor with einx ... rearraging a dimension to be
# treated as complex numbers ... 

In [ ]:
einx.rearrange("... (c d) -> ... d c",tt6, c=2)

In [ ]:
# actually, using a 4-D vector may leave stuff a little bit ambiguous when you try to follow 
# exactly how these indices are being rearranged ... 
#  lets start over with a 6-D vector 
# (aside ... using the term dimensions for the array entries gets a bit confusing, since we 
# often use the term 2D array, or 3D-array ... meaning a matrix with 2 indices, or 3 indices)
# which makes one appreciate using the nomenclature "tensor of rank 1", tensor of rank 2, tensor of rank 3, etc.
# which is not ambiguous, and you can feel free then to say ?perhaps? that there are say 4 dimensions
# along the axis of the first rank?

# Define a nice, simple rank 1 tensor to work with
tt6 = torch.arange(6)
# rearrange the rank 1 tensor to be of rank 2
tt6_reshaped = einx.rearrange("... (d c) -> ... d c",tt6, c=2)
# there is a fair bit going on in the line above:
# - for one thing there is an ellipsis starting off in the quotes, 
# this is saying that the "dimension" I wish to access (with the (d, c))
# is at the very end of the tensor.  If we had instead said "(d, c) ... -> ... d c"
# we'd've been saying to rearrange on the first coordinate.
# But all this is kinda moot here, since there is only one coordinate in our tt6 array.
# It's nice that einx seems to not really care if we put an ellipsis there when
# it isn't really needed, i guess it treats a list of 0, 1, ...N coordinates to 
# -- searhcing for word -- "broadcast over"? the same
# 
# anyhow, we can see the different reshapings by printing the output of the following  
tt6_reshaped = einx.rearrange("... (c d) -> ... d c",tt6, c=2)
tt6_reshaped = einx.rearrange("... (c d) -> ... c d",tt6, c=2)
tt6_reshaped = einx.rearrange("... (d c) -> ... c d",tt6, c=2)
# there is something that remonds me of. agroup of order 4 in the previous collection of reshapings 
# ... 
tt6_reshaped = einx.rearrange("... (d c) -> ... c d",tt6, c=2, d=2)
tt6_reshaped = einx.rearrange("... (d c) -> ... c d",tt6, c=2, d=3)
torch.view_as_complex(tt6_reshaped)

# (b, s, h, ...) "batch", "sequence", "head", 
# The variable theta can be indexed by i, k.
# i indexes the positions, and has as many values as the subspace has dimensons.